> **Disclaimer:** This workshop is provided for educational and informational purposes only. The sample code and configurations are not intended for production use. Please review and adapt them according to your organization's security and compliance requirements. AWS service charges may apply for resources created during this workshop. Video content used in this workshop is sourced from publicly available materials and is attributed where applicable.

# Approach 3: Intent-based Dynamic Query Routing

This notebook implements anchor-based dynamic routing from Section 4.3 of the whitepaper.

Instead of fixed weights, the system **automatically determines per-query weights** by measuring how closely the query aligns with modality-specific "anchors" using the embedding space itself.

**Formula:** `(w_v, w_a, w_t) = softmax(α * sim(E_query, [E_AncV, E_AncA, E_AncT]))` where α=10

In [ ]:
import json
import numpy as np
import boto3
from botocore.config import Config
import matplotlib.pyplot as plt
from collections import defaultdict

# Auto-load configuration from 01_setup_and_embedding.ipynb
with open("config.json") as f:
    config = json.load(f)

REGION = config["REGION"]
S3_BUCKET = config["S3_BUCKET"]
MODEL_ID_SYNC = config["MODEL_ID_SYNC"]
VECTOR_BUCKET = config["VECTOR_BUCKET"]
DATA_S3_PREFIX = config["DATA_S3_PREFIX"]
RESULTS_S3_PREFIX = config["RESULTS_S3_PREFIX"]

print(f"✅ Configuration loaded from config.json")
print(f"   S3 Bucket: {S3_BUCKET}")
print(f"   Vector Bucket: {VECTOR_BUCKET}")

bedrock = boto3.client("bedrock-runtime", region_name=REGION, config=Config(signature_version='v4'))
s3 = boto3.client("s3", region_name=REGION)
s3v = boto3.client("s3vectors", region_name=REGION)

def fmt_time(sec):
    """Convert seconds to mm:ss format"""
    m, s = divmod(sec, 60)
    return f"{int(m):02d}:{s:04.1f}"

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def embed_text(query):
    response = bedrock.invoke_model(
        modelId=MODEL_ID_SYNC,
        body=json.dumps({"inputType": "text", "text": {"inputText": query}})
    )
    return json.loads(response["body"].read())["data"][0]["embedding"]

def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum()

print("Setup complete")

## 1. Load Separate Modality Indices

In [ ]:
# Load embeddings from S3
indices = {}
for modality in ["visual", "audio", "transcription"]:
    s3_key = f"{DATA_S3_PREFIX}embeddings_{modality}.json"
    obj = s3.get_object(Bucket=S3_BUCKET, Key=s3_key)
    indices[modality] = json.loads(obj["Body"].read().decode("utf-8"))
    print(f"{modality}: {len(indices[modality])} segments")

## 2. Define and Embed Routing Anchors

Anchors are text descriptions that represent the semantic intent of each modality.

In [ ]:
ANCHORS = {
    "visual": "This document contains content about the visual elements, scenes, objects, and imagery in the video.",
    "audio": "This document contains content about the non-speech audio elements such as music, sound effects, and ambient sounds in the video.",
    "transcription": "This document contains content about the spoken words, dialogue, narration, and speech transcribed from the video."
}

ALPHA = 10

# Embed anchors
anchor_embeddings = {}
for modality, text in ANCHORS.items():
    anchor_embeddings[modality] = embed_text(text)
    print(f"Embedded anchor for {modality}: dim={len(anchor_embeddings[modality])}")

print(f"\nTemperature (alpha): {ALPHA}")
print(f"\nAnchor texts:")
for m, t in ANCHORS.items():
    print(f"  {m}: \"{t}\"")

## 3. Dynamic Weight Calculation

For each query, compute cosine similarity with each anchor, then apply softmax with temperature α.

In [ ]:
def compute_dynamic_weights(query_text, anchor_embeddings, alpha=ALPHA):
    """Compute per-query modality weights using anchor similarity (Equation 4)"""
    q_emb = embed_text(query_text)
    
    similarities = {}
    for modality, anc_emb in anchor_embeddings.items():
        similarities[modality] = cosine_similarity(q_emb, anc_emb)
    
    # Apply softmax with temperature
    sim_array = np.array([similarities["visual"], similarities["audio"], similarities["transcription"]])
    weights_array = softmax(alpha * sim_array)
    
    weights = {
        "visual": float(weights_array[0]),
        "audio": float(weights_array[1]),
        "transcription": float(weights_array[2])
    }
    
    return weights, similarities, q_emb

# Demo with a few queries
demo_queries = [
    "a player scoring a goal with a powerful shot",
    "loud crowd chanting in the stadium",
    "the commentator explains the offside rule",
    "soccer players celebrating after a victory",
    "the announcer discussing match statistics while fans cheer"
]

print("Dynamic Weights Demo:")
print("=" * 80)
for q in demo_queries:
    weights, sims, _ = compute_dynamic_weights(q, anchor_embeddings)
    dominant = max(weights, key=weights.get)
    print(f"\nQuery: \"{q}\"")
    print(f"  Anchor similarities: V={sims['visual']:.4f} A={sims['audio']:.4f} T={sims['transcription']:.4f}")
    print(f"  Dynamic weights:     V={weights['visual']:.4f} A={weights['audio']:.4f} T={weights['transcription']:.4f}")
    print(f"  Dominant modality:   {dominant} ({weights[dominant]*100:.1f}%)")

## 4. Search with Dynamic Routing

In [ ]:
def search_with_dynamic_routing(query_text, anchor_embeddings, alpha=ALPHA, top_k=5):
    """Multi-vector search with dynamically computed weights via S3 Vectors"""
    weights, similarities, q_emb = compute_dynamic_weights(query_text, anchor_embeddings, alpha)
    
    # Search each modality index in S3 Vectors
    combined = defaultdict(lambda: {"scores": {}, "startSec": 0, "endSec": 0})
    
    for modality in weights:
        response = s3v.query_vectors(
            vectorBucketName=VECTOR_BUCKET,
            indexName=modality,
            topK=50,
            queryVector={"float32": [float(x) for x in q_emb]},
            returnMetadata=True,
            returnDistance=True
        )
        for v in response["vectors"]:
            key = (round(v["metadata"]["startSec"], 1), round(v["metadata"]["endSec"], 1))
            combined[key]["scores"][modality] = v["distance"]
            combined[key]["startSec"] = v["metadata"]["startSec"]
            combined[key]["endSec"] = v["metadata"]["endSec"]
    
    # Weighted combination with dynamic weights
    results = []
    for key, data in combined.items():
        score = sum(weights[m] * data["scores"].get(m, 0) for m in weights)
        results.append({
            "startSec": data["startSec"],
            "endSec": data["endSec"],
            "combined_score": score,
            "visual_score": data["scores"].get("visual", 0),
            "audio_score": data["scores"].get("audio", 0),
            "transcription_score": data["scores"].get("transcription", 0)
        })
    
    results.sort(key=lambda x: x["combined_score"], reverse=True)
    return results[:top_k], weights, similarities

## 5. Test: Compare Fixed vs Dynamic Weights

In [ ]:
TEST_QUERIES = [
    {"query": "a player kicking the ball towards the goal", "intent": "visual"},
    {"query": "crowd cheering and celebrating loudly", "intent": "audio"},
    {"query": "the commentator describes the player's performance", "intent": "transcription"},
    {"query": "a goal being scored with the stadium erupting", "intent": "visual+audio"},
    {"query": "the announcer explaining a foul while the crowd boos", "intent": "mixed"},
    {"query": "players running on the soccer field", "intent": "visual"},
    {"query": "referee whistle blowing during the match", "intent": "audio"},
    {"query": "the commentator discusses the team's strategy", "intent": "transcription"},
]

FIXED_WEIGHTS = {"visual": 0.8, "audio": 0.1, "transcription": 0.05}

all_dynamic_results = {}
all_weight_data = []

for item in TEST_QUERIES:
    query = item["query"]
    intent = item["intent"]
    
    results, dyn_weights, sims = search_with_dynamic_routing(query, anchor_embeddings)
    all_dynamic_results[query] = results
    all_weight_data.append({
        "query": query,
        "intent": intent,
        "dynamic_weights": dyn_weights,
        "fixed_weights": FIXED_WEIGHTS,
        "anchor_similarities": sims
    })
    
    print(f"\n{'='*70}")
    print(f"Query: \"{query}\" (intent: {intent})")
    print(f"  Fixed weights:   V={FIXED_WEIGHTS['visual']:.2f} A={FIXED_WEIGHTS['audio']:.2f} T={FIXED_WEIGHTS['transcription']:.2f}")
    print(f"  Dynamic weights: V={dyn_weights['visual']:.4f} A={dyn_weights['audio']:.4f} T={dyn_weights['transcription']:.4f}")
    print(f"  Top 3 results:")
    for i, r in enumerate(results[:3], 1):
        print(f"    {i}. [{fmt_time(r['startSec'])}-{fmt_time(r['endSec'])}] score={r['combined_score']:.4f}")

## 6. Visualize Dynamic Weight Distributions

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle("Dynamic Weights per Query (Intent-based Routing)", fontsize=14, fontweight='bold')

colors = {'visual': '#2196F3', 'audio': '#FF9800', 'transcription': '#4CAF50'}

for idx, data in enumerate(all_weight_data):
    row = idx // 4
    col = idx % 4
    ax = axes[row][col]
    
    modalities = list(data["dynamic_weights"].keys())
    dyn_vals = [data["dynamic_weights"][m] for m in modalities]
    fix_vals = [data["fixed_weights"][m] for m in modalities]
    
    x = np.arange(len(modalities))
    width = 0.35
    
    bars1 = ax.bar(x - width/2, fix_vals, width, label='Fixed', alpha=0.5, color='gray')
    bars2 = ax.bar(x + width/2, dyn_vals, width, label='Dynamic', 
                   color=[colors[m] for m in modalities])
    
    ax.set_ylabel('Weight')
    ax.set_title(f'"{ data["query"][:30]}..."' if len(data["query"]) > 30 else f'"{data["query"]}"', fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels(['V', 'A', 'T'])
    ax.set_ylim(0, 1.0)
    ax.legend(fontsize=7)

plt.tight_layout()

# Save plot to S3
import io
buf = io.BytesIO()
plt.savefig(buf, format='png', dpi=150, bbox_inches='tight')
buf.seek(0)

s3_key = f"{RESULTS_S3_PREFIX}dynamic_weights_comparison.png"
s3.put_object(
    Bucket=S3_BUCKET,
    Key=s3_key,
    Body=buf.getvalue(),
    ContentType="image/png"
)
print(f"Visualization saved to s3://{S3_BUCKET}/{s3_key}")

plt.show()

## 7. Key Insight: Query-Adaptive Behavior

The intent-based routing system **automatically adjusts weights** based on query semantics:
- Visual queries get high visual weight
- Audio queries get high audio weight
- Transcription queries get high transcription weight
- Mixed queries get balanced weights

This is **deterministic** (same query always gives same weights) and **explainable** (you can inspect anchor similarities).

In [ ]:
results_data = {
    "approach": "intent_based_dynamic_routing",
    "alpha": ALPHA,
    "anchors": ANCHORS,
    "weight_data": all_weight_data,
    "results": {q: r for q, r in all_dynamic_results.items()}
}

s3_key = f"{RESULTS_S3_PREFIX}intent_routing_results.json"
s3.put_object(
    Bucket=S3_BUCKET,
    Key=s3_key,
    Body=json.dumps(results_data, indent=2, default=float),
    ContentType="application/json"
)

print(f"Results saved to s3://{S3_BUCKET}/{s3_key}")

---

## Summary: Intent-based Dynamic Routing Approach

### Architecture
```
[Preparation: Anchor embeddings — once at service startup]
  "visual elements, scenes, objects..."       -> embed -> anchor_visual (512-dim)
  "non-speech audio, music, sound effects..." -> embed -> anchor_audio (512-dim)
  "spoken words, dialogue, narration..."      -> embed -> anchor_trans (512-dim)

[Query processing — per query]
  "crowd cheering and celebrating loudly" -> embed -> query_vector
       |
       |-- 1. Weight calculation (cosine + softmax)
       |     cosine(query, anchor_visual) = 0.72
       |     cosine(query, anchor_audio)  = 0.81
       |     cosine(query, anchor_trans)  = 0.68
       |     -> softmax(alpha=10 x [0.72, 0.81, 0.68])
       |     -> weights: [V=0.24, A=0.60, T=0.16]
       |
       |-- 2. Search with dynamic weights
             visual index search       x 0.24
             audio index search        x 0.60  <- audio gets the highest weight
             transcription index search x 0.16
             -> aggregate -> final ranking
```

### How Automatic Weight Calculation Works

**Why can embedding space distance determine intent?**

Marengo places text and video in the same 512-dim space.
So the text "crowd cheering" is naturally positioned close to
the audio-related anchor text in the embedding space.

**Role of alpha (temperature parameter):**

Cosine similarity differences are usually small (0.72 vs 0.81 vs 0.68).
Multiplying by alpha amplifies the differences, and applying softmax produces a clear weight distribution.

```
alpha=1:   [0.32, 0.37, 0.31]  -> nearly uniform (insufficient amplification)
alpha=10:  [0.24, 0.60, 0.16]  -> clear differentiation (recommended)
alpha=20:  [0.10, 0.82, 0.08]  -> nearly single-modality selection
```

[Example]
Step 1: Multiply by alpha

  Multiply original anchor similarity by alpha=10 to amplify differences.

  alpha x 0.2575 = 2.575
  alpha x 0.1102 = 1.102
  alpha x 0.2082 = 2.082

  Step 2: Subtract max for numerical stability

  max = 2.575

  2.575 - 2.575 =  0.000
  1.102 - 2.575 = -1.473
  2.082 - 2.575 = -0.493

  Step 3: Compute exp

  exp( 0.000) = 1.0000
  exp(-1.473) = 0.2291
  exp(-0.493) = 0.6107

  Step 4: Divide by sum (normalize)

  sum = 1.0000 + 0.2291 + 0.6107 = 1.8398

  w_v = 1.0000 / 1.8398 = 0.5435  ->  54.3%
  w_a = 0.2291 / 1.8398 = 0.1245  ->  12.5%
  w_t = 0.6107 / 1.8398 = 0.3320  ->  33.2%


### Improvements over Notebook 03 (Fixed Weights)

| Query | 03 Fixed | 04 Dynamic |
|---|---|---|
| "players running on the soccer field" | V=0.80 A=0.10 T=0.05 | V=**~0.85** A=~0.08 T=~0.07 |
| "crowd cheering and celebrating loudly" | V=0.80 A=0.10 T=0.05 | V=~0.24 A=**~0.60** T=~0.16 |
| "the commentator discusses team's strategy" | V=0.80 A=0.10 T=0.05 | V=~0.09 A=~0.15 T=**~0.76** |

-> Notebook 03 always uses visual 80%, Notebook 04 **auto-adjusts based on query intent**

### API Call Cost

| Item | Count | Timing |
|---|---|---|
| Anchor embeddings | 3 (fixed) | Once at service initialization |
| Query embedding | 1/query | Per search request |
| Cosine similarity | 3/query | Local computation (near-zero cost) |
| S3 Vectors search | 3/query | visual + audio + transcription indexes |

-> Additional cost vs. Notebook 03 is **3 cosine similarities + 1 softmax** (negligible)

### Properties of This Approach
- **Automatic**: No need to manually specify weights per query
- **Deterministic**: Same query always produces the same weights (no randomness)
- **Explainable**: Anchor similarities show why specific weights were assigned
- **No LLM required**: Query intent classification does not require an LLM call
- **Reversible**: Original modality embeddings are preserved, so anchor text or alpha can be changed for re-experimentation

### Full Progression Summary (02 -> 03 -> 04)
```
02 Fused:    Merged at storage, fixed weights, irreversible   -> Identify limitations
03 Fixed:    Merged at query time, fixed weights, reversible   -> Debuggable but still fixed
04 Dynamic:  Merged at query time, auto weights, reversible    -> Adapts to query intent
```